Paths and reusable definitions were refactored. Stored outputs below are historical, not a fresh execution. See `docs/WORKFLOW.md` at the repository root before rerunning; several stages overwrite results or append Optuna trials.


In [ ]:
# Locate this checkout from the repository root or any notebook subdirectory.
from pathlib import Path
import sys

_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "src" / "steam_recommender" / "paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter inside the repository checkout.")
sys.path.insert(0, str(_root / "src"))
from steam_recommender import paths as _paths
if Path(_paths.__file__).resolve() != (_root / "src/steam_recommender/paths.py").resolve():
    raise RuntimeError("A different checkout is cached; restart the kernel.")
from steam_recommender.paths import project_root, project_path


In [1]:
import pandas as pd
import numpy as np

In [2]:
users_items = pd.read_csv(str(project_path("data/raw/users_items.csv")))
print("% of zero playtime entries: {:.2f}".format((users_items['playtime_forever'] == 0).mean() * 100))
print(users_items.head())

% of zero playtime entries: 36.25
             user_id  item_id                  item_name  playtime_forever
0  76561197970982479       10             Counter-Strike                 6
1  76561197970982479       20      Team Fortress Classic                 0
2  76561197970982479       30              Day of Defeat                 7
3  76561197970982479       40         Deathmatch Classic                 0
4  76561197970982479       50  Half-Life: Opposing Force                 0


In [3]:
from steam_recommender.preprocessing import prepare_interactions

# One pass: cap playtime, filter games, then filter users.
users_items_filtered, cold_start_users = prepare_interactions(users_items)

print(users_items_filtered.head())
print(cold_start_users.head())

             user_id  item_id                  item_name  playtime_forever
0  76561197970982479       10             Counter-Strike                 6
1  76561197970982479       20      Team Fortress Classic                 0
2  76561197970982479       30              Day of Defeat                 7
3  76561197970982479       40         Deathmatch Classic                 0
4  76561197970982479       50  Half-Life: Opposing Force                 0
                 user_id  item_id                         item_name  \
16543  76561198157999555     4000                       Garry's Mod   
16544  76561198157999555   226320                Marvel Heroes 2016   
16545  76561198157999555      730  Counter-Strike: Global Offensive   
16623  76561198074200011   326460                   ShellShock Live   
16624  76561198074200011      730  Counter-Strike: Global Offensive   

       playtime_forever  
16543               488  
16544                 5  
16545             25194  
16623              

In [4]:
from steam_recommender.splitting import train_test_split_first


users_items_filtered_train_val, users_items_filtered_test = train_test_split_first(users_items_filtered, test_frac=0.1, seed=42)


In [5]:
import os

save_path = str(project_path("data/processed"))

os.makedirs(save_path, exist_ok=True)


# save splits
users_items_filtered_train_val.to_csv(
    os.path.join(save_path, "users_items_filtered_train_val.csv"),
    index=False
)

users_items_filtered_test.to_csv(
    os.path.join(save_path, "users_items_filtered_test.csv"),
    index=False
)

# save cold start unchanged
cold_start_users.to_csv(
    os.path.join(save_path, "cold_start_users.csv"),
    index=False
)

print("Saved:")
print("train_val:", users_items_filtered_train_val.shape)
print("test:", users_items_filtered_test.shape)
print("cold_start_users:", cold_start_users.shape)

Saved:
train_val: (4591379, 4)
test: (541231, 4)
cold_start_users: (16902, 4)
